# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [9]:
# Your code goes here

import pandas as pd
import numpy as np

# 1. Load the raw datasets
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)


# 2. Function to standardize headers before combining
def standardize_headers(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    rename_dict = {'st': 'state'}
    return df.rename(columns=rename_dict)


print("🔄 Standardizing column headers...")
df1_cols = standardize_headers(df1)
df2_cols = standardize_headers(df2)
df3_cols = standardize_headers(df3)


print("🔗 Concatenating datasets...")
combined_df = pd.concat([df1_cols, df2_cols, df3_cols], axis=0, ignore_index=True)


# 3. Global data cleaning and formatting pipeline
def clean_insurance_data(df):
    df = df.copy()  
    
    # --- Clean 'gender' variations ---
    if 'gender' in df.columns:
        gender_mapping = {
            'Femal': 'F', 'female': 'F', 'Female': 'F', 'F': 'F',
            'Male': 'M', 'male': 'M', 'M': 'M'
        }
        df['gender'] = df['gender'].replace(gender_mapping)
        df['gender'] = df['gender'].fillna('U')  
        
    # --- Clean 'customer_lifetime_value' ---
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = df['customer_lifetime_value'].astype(str).str.replace('%', '')
        df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
        
    # --- Clean 'state' text variations ---
    if 'state' in df.columns:
        state_mapping = {
            'AZ': 'Arizona', 
            'Cali': 'California', 
            'WA': 'Washington'
        }
        df['state'] = df['state'].replace(state_mapping)
        
    # --- Clean 'number_of_open_complaints' safely without crashing ---
    if 'number_of_open_complaints' in df.columns:
        # Step A: Convert to string type to avoid float mismatches
        df['number_of_open_complaints'] = df['number_of_open_complaints'].astype(str)
        
        # Step B: Native Pandas string split - extract index 1 if '/' exists, otherwise it keeps the text
        # (e.g. '1/0/00' -> '0')
        split_complaints = df['number_of_open_complaints'].str.split('/')
        df['number_of_open_complaints'] = split_complaints.apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else x[0] if isinstance(x, list) else x)
        
        # Step C: Force to numeric, reverting 'nan' strings into true Null values (NaN)
        df['number_of_open_complaints'] = pd.to_numeric(df['number_of_open_complaints'], errors='coerce')

    # --- Remove exact duplicate rows ---
    df = df.drop_duplicates()
    
    return df


# 4. Execute the cleaning pipeline
print("🧼 Running data cleaning functions...")
final_cleaned_df = clean_insurance_data(combined_df)

# 5. Reset the final index to ensure a sequential integer index
final_cleaned_df = final_cleaned_df.reset_index(drop=True)

# 6. Verify results
print("\n✅ Data cleaning complete!")
print("-" * 50)
print(final_cleaned_df.info())
print("-" * 50)

# Quick look at the cleaned values
if 'number_of_open_complaints' in final_cleaned_df.columns:
    print("Unique values in 'number_of_open_complaints':", final_cleaned_df['number_of_open_complaints'].unique())


🔄 Standardizing column headers...
🔗 Concatenating datasets...
🧼 Running data cleaning functions...

✅ Data cleaning complete!
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 9135 entries, 0 to 9134
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9134 non-null   str    
 1   state                      9134 non-null   str    
 2   gender                     9135 non-null   str    
 3   education                  9134 non-null   str    
 4   customer_lifetime_value    9127 non-null   float64
 5   income                     9134 non-null   float64
 6   monthly_premium_auto       9134 non-null   float64
 7   number_of_open_complaints  9134 non-null   float64
 8   policy_type                9134 non-null   str    
 9   vehicle_class              9134 non-null   str    
 10  total_claim_amount         9134 non-null   float64

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [18]:
# Your code goes here

import pandas as pd
import numpy as np

# 1. Load the new marketing customer analysis dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)
print("Fetching the consolidated marketing customer analysis dataset...")
customer_df = pd.read_csv(url)

# 2. Standardize column formatting immediately (PE8 / snake_case guidelines)
customer_df.columns = customer_df.columns.str.lower().str.replace(' ', '_')

# 3. Handle specific data types (e.g., convert date representations to datetime)
if 'effective_to_date' in customer_df.columns:
    customer_df['effective_to_date'] = pd.to_datetime(customer_df['effective_to_date'], errors='coerce')

# 4. Display high-level structure & structural dimensions
print(" Dataset Metadata Summary:")
print("-" * 50)
print(f"Total Rows:    {customer_df.shape[0]}")
print(f"Total Columns: {customer_df.shape[1]}")
print("-" * 50)
customer_df.info()

# 5. Isolate numerical and categorical blocks for further analysis
numerical_df = customer_df.select_dtypes(include=[np.number])
categorical_df = customer_df.select_dtypes(exclude=[np.number, 'datetime64[ns]'])

print(f"\n🔢 Detected {numerical_df.shape[1]} Numerical features.")
print(f"🔤 Detected {categorical_df.shape[1]} Categorical features.")
# Extract just the name of the month from the date
customer_df['month'] = customer_df['effective_to_date'].dt.month_name()

# Find out how many days have passed between today and when the policy ends
customer_df['days_left'] = customer_df['effective_to_date'] - pd.Timestamp.now()


df.columns

Fetching the consolidated marketing customer analysis dataset...
 Dataset Metadata Summary:
--------------------------------------------------
Total Rows:    10910
Total Columns: 27
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   unnamed:_0                     10910 non-null  int64         
 1   customer                       10910 non-null  str           
 2   state                          10910 non-null  str           
 3   customer_lifetime_value        10910 non-null  float64       
 4   response                       10910 non-null  str           
 5   coverage                       10910 non-null  str           
 6   education                      10910 non-null  str           
 7   effective_to_date              10910 non-null  datetime64[us]

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type', 'month'],
      dtype='str')

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [21]:
# Standardize columns to match your exact snake_case index layout
df.columns = df.columns.str.lower().str.replace(' ', '_')

# --- Pivot Table 1: Total Revenue (Sum of Monthly Premiums) by Sales Channel ---
sales_channel_pivot = df.pivot_table(
    index='sales_channel', 
    values='monthly_premium_auto', 
    aggfunc='sum'
).round(2)

print("=== Pivot Table 1: Total Revenue by Sales Channel ===")
print(sales_channel_pivot)
print("\n" + "="*50 + "\n")

# --- Pivot Table 2: Average Customer Lifetime Value (CLV) by Gender & Education ---
clv_pivot = df.pivot_table(
    index='education', 
    columns='gender', 
    values='customer_lifetime_value'#,aggfunc='mean'
).round(2)

print("=== Pivot Table 2: Average CLV by Gender & Education ===")
print(clv_pivot)


=== Pivot Table 1: Total Revenue by Sales Channel ===
               monthly_premium_auto
sales_channel                      
Agent                        386335
Branch                       280953
Call Center                  197970
Web                          151511


=== Pivot Table 2: Average CLV by Gender & Education ===
gender                      F        M
education                             
Bachelor              7874.27  7703.60
College               7748.82  8052.46
Doctor                7328.51  7415.33
High School or Below  8675.22  8149.69
Master                8157.05  8168.83


1. Revenue Performance by Sales ChannelUsing Total Monthly Premiums as our primary proxy for active sales volume, the revenue distribution across channels breaks down as follows:
   

👥 The Human-Centric Powerhouses: Agent is our clear market leader, single-handedly driving 386,335, followed by physical Branches at 280,953. Combined, these two traditional channels generate nearly 70% of our total revenue. This shows that when purchasing complex financial products like insurance, consumers deeply value direct relationship-based guidance and customized advice.

💻 The Underutilized Margin Engine (Web): At 151,511, the Web channel brings in the least revenue. However, digital transactions carry the lowest customer acquisition and administrative overhead. This gap reveals a clear opportunity: we need to improve our online conversion flows and shift lower-risk or transactional customers to digital self-service.

📞 The Support Core (Call Center): Moving 197,970 in volume, the call center remains a stable, essential baseline channel, capturing customers who want a human touch without driving to a brick-and-mortar office.

2. Customer Lifetime Value (CLV) TrendsAnalyzing the average long-term financial weight of our policyholders across demographic segments highlights several key trends:

🎓 The Education Paradox: Stability Rules

High School or Below Leads Value: Interestingly, the highest overall individual baseline values come from the High School or Below cohort (Females: 8,675.22; Males: 8,149.69). In insurance modeling, this typically happens because these clients are highly brand-loyal, hold multi-car policies longer, and exhibit lower contract shopping/churn rates compared to highly mobile university graduates.

The High-Income Anchor: Post-graduates with a Master degree rank as the second-most valuable cohort (~8,160 average across genders), driven by high household net worth and umbrella policy needs.

The Academic Dip: Holders of Doctor degrees present our lowest average lifetime value (7,328.51 for Females; 7,415.33 for Males). While highly educated, this segment is highly price-sensitive and frequently utilizes online rate aggregators to churn to lower competitors.

🚻 Gender Performance is Even

Demographic Equality:Gender does not create major statistical divides in long-term account values. The difference within any education tier is negligible—for example, only an 11.78 difference between Male and Female Master's degree holders. Gender should never be used to filter target acquisition ad spend.

🛠️ Strategic Recommendations for the Department

Protect the Core: Maintain our localized Agent enablement programs, but equip them with tools to cross-sell standard umbrella or home packages specifically to their Master and High School baseline customer files.

Optimize Digital Acquisition: Launch a targeted web optimization campaign aimed at mid-tier accounts.
Shifting customers who behave like the Doctor cohort away from expensive Agent loops and into digital self-service funnels will instantly boost corporate profit margins.






## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here